In [25]:
from ddgs import DDGS

def ddg_search(query: str, max_results: int = 5):
    with DDGS() as ddgs:
        # backend 可选：lite / html / news
        return list(ddgs.text(query, max_results=max_results, backend="lite"))

print(ddg_search("LangChain 是什么", 3))

[{'title': '什么是LangChain？ –LangChain简介 – AWS', 'href': 'https://aws.amazon.com/cn/what-is/langchain/', 'body': '为什么LangChain很重要？LangChain是一个开源框架，用于构建基于大型语言模型（LLM）的应用程序。'}, {'title': '什么是LangChain？| IBM', 'href': 'https://www.ibm.com/cn-zh/think/topics/langchain', 'body': 'LangChain由 Harrison Chase 于 2022 年 10 月推出，之后迅速脱颖而出：截至 2023 年 6 月，LangChain是Github 上增长最快的开源项目。'}, {'title': 'LangChain是什麼？ AI開發者必須了解的LLM開源框架 - ALPHA Camp', 'href': 'https://tw.alphacamp.co/blog/langchain-intro', 'body': 'LangChain是一個旨在為開發者提供一套工具和接口，以便更容易、更有效地利用大型語言模型（LLM）的開源框架，專注於情境感知和推理。'}]


In [26]:
from langchain.tools import tool
from pydantic import BaseModel, Field
from datetime import datetime
import json

class WeatherInput(BaseModel):
    location: str = Field(description="城市或县区，比如北京市、杭州市、余杭区等。")

@tool("get_current_weather", args_schema=WeatherInput)
def get_current_weather(location: str) -> str:
    """当你想查询指定城市的天气时非常有用。"""
    return f"{location}今天是雨天。"

@tool("get_current_time")
def get_current_time() -> str:
    """当你想知道现在的时间时非常有用。"""
    return f"当前时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}。"

#定义输入的内容，包括query和max_results，给LLM 看的 JSON Schema。
class WebSearchInput(BaseModel):
    query: str = Field(description="要搜索的关键词，例如：LangChain 是什么")
    max_results: int = Field(default=3, description="返回结果条数，建议 3-5")

#实际调用搜索引擎的部分
def ddg_search(query: str, max_results: int = 3):
    with DDGS() as ddgs:
        # 你当前已验证可用
        return list(ddgs.text(query, max_results=max_results, backend="lite"))

#封装为 LangChain Tool
@tool("web_search", args_schema=WebSearchInput)
def web_search_tool(query: str, max_results: int = 3) -> str:
    """通用网络搜索工具（国内可用，基于 DuckDuckGo / ddgs）。返回 top-k 的标题、链接、摘要。"""
    results = ddg_search(query, max_results=max_results)
    return json.dumps(results, ensure_ascii=False) #返回字符串，因为tool必须返回str

In [27]:
import os
from langchain_deepseek import ChatDeepSeek

# 确保环境变量存在，或直接在下面传 api_key 参数（如果该版本支持）
#assert os.getenv("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

llm = ChatDeepSeek(
    model="deepseek-chat",   # DeepSeek-V3: 支持 tools/structured output
    temperature=0,
    api_key="sk-3d4dbf63d7704840976b05ddbb9957a7"
)
tools = [get_current_time, get_current_weather, web_search_tool]
llm_with_tools = llm.bind_tools(tools)

In [29]:
VEHICLE_DB = {
    ("Model Y", 2024): {
        "range": "CLTC 545-688km（不同版本不同）",
        "battery": "磷酸铁锂/三元（因版本/产地而异）",
        "charging": "支持快充，峰值功率因版本而异",
        "dimensions": "约4750×1921×1624mm（不同版本略有差异）",
    },
    ("Model 3", 2024): {
        "range": "CLTC 606-713km（不同版本不同）",
        "battery": "磷酸铁锂/三元（因版本而异）",
    },
}

In [54]:
from typing import Optional,List,Dict
from pydantic import BaseModel,Field

class VehicleSpecSchema(BaseModel):
    model:str=Field(decription="车型名称，比如Model X,Model Y,Model Z等")
    year:Optional[int]=Field(description="年份，例如2024，2025，2026等")
    fields:Optional[List[str]]=Field(description="想要查询的字段，例如['range','battery']")

/tmp/ipykernel_2868812/534316185.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  model:str=Field(decription="车型名称，比如Model X,Model Y,Model Z等")


In [55]:
@tool('get_vehicle_spec',args_schema=VehicleSpecSchema)
def get_vehicle_spec(model:str,year:Optional[int]=None,fields:Optional[List[str]]=None)->Dict:
    """
    利用本地字典，查询车辆的参数
    """
    y=year if year is not None else 2024
    key=(model,y)
    info=VEHICLE_DB.get(key)

    if not info :
        return {
            "model":model,
            "year":y,
            "found":False,
            "data":{},
            "hint":"暂无该车型或者该年份信息"
        }
    if fields:
        data={f:info.get(f,"该字段暂无信息") for f in fields}
    else:
        data=info
    return {"model": model, "year": y, "found": True, "data": data}

In [56]:
tools = [get_current_time, get_current_weather, web_search_tool,get_vehicle_spec]
llm_with_tools = llm.bind_tools(tools)

In [57]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {t.name: t for t in tools}

def call_with_messages(content,llm):
    messages = [HumanMessage(content=content)]
    print("-" * 60)

    i = 1
    ai = llm.invoke(messages)
    print(f"\n第{i}轮大模型输出：{ai}\n")

    while ai.tool_calls:
        messages.append(ai)

        for call in ai.tool_calls:
            tool_name = call["name"]
            tool_args = call.get("args", {}) or {}
            tool_id = call["id"]

            result = tool_map[tool_name].invoke(tool_args)

            print(f"工具调用：{tool_name} args={tool_args}")
            print(f"工具输出：{result}\n")
            print("-" * 60)

            messages.append(
                ToolMessage(
                    name=tool_name,
                    tool_call_id=tool_id,
                    content=str(result)
                )
            )

        i += 1
        ai = llm_with_tools.invoke(messages)
        print(f"\n第{i}轮大模型输出：{ai}\n")

    print(f"最终答案：{ai.content}")

#call_with_messages("杭州和北京天气怎么样？现在几点了？另外 LangChain 是什么？",llm_with_tools)

In [52]:
call_with_messages("请查一下 2024 款 Model Y 的续航和电池信息，并告诉我现在几点。",llm_with_tools)

------------------------------------------------------------

第1轮大模型输出：content='我来帮您查询2024款Model Y的续航和电池信息，并获取当前时间。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 670, 'total_tokens': 790, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640}, 'prompt_cache_hit_tokens': 640, 'prompt_cache_miss_tokens': 30}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'b07d5607-4538-4aea-a267-07f96086ddb6', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cb303-4f96-7502-99cf-87e8ccb9ef0e-0' tool_calls=[{'name': 'get_vehicle_spec', 'args': {'model': 'Model Y', 'year': 2024, 'fields': ['range', 'battery']}, 'id': 'call_00_IA80UcQrCcd2tATVGzZOeFvZ', 'type': 'tool_call'}, {'name': 'get_current_time', 'args': {}, 'id': 'call_01_1nilsl6iHe37KUx6p1E2fLBF', 'type': 'tool_call'}] invalid_tool_calls

In [58]:
call_with_messages("请查一下 2026 款 Model Z 的续航和电池信息",llm_with_tools)

------------------------------------------------------------

第1轮大模型输出：content='我来帮您查询2026款Model Z的续航和电池信息。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 633, 'total_tokens': 733, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 448}, 'prompt_cache_hit_tokens': 448, 'prompt_cache_miss_tokens': 185}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '5b43da7e-0356-4556-8ac0-14af9de997ce', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cb305-164d-7560-bb8c-b8cdc73db0f9-0' tool_calls=[{'name': 'get_vehicle_spec', 'args': {'model': 'Model Z', 'year': 2026, 'fields': ['range', 'battery']}, 'id': 'call_00_vs5fp0yDbooCpzvKHkbwk1Ka', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 633, 'output_tokens': 100, 'total_tokens': 733, 'input_token_details': {'cac

In [59]:
call_with_messages("请查一下 2024 款 Model Y 的充电参数和车辆尺寸",llm_with_tools)

------------------------------------------------------------

第1轮大模型输出：content='我来帮您查询2024款Model Y的充电参数和车辆尺寸信息。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 634, 'total_tokens': 736, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 576}, 'prompt_cache_hit_tokens': 576, 'prompt_cache_miss_tokens': 58}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'a5bab947-c08e-4789-b891-2bc160b5ab51', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cb306-b449-7bf1-9381-9b215168ef63-0' tool_calls=[{'name': 'get_vehicle_spec', 'args': {'model': 'Model Y', 'year': 2024, 'fields': ['charging', 'dimensions']}, 'id': 'call_00_XTgBmD3DkJE7FsBaL5lsHuLN', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 634, 'output_tokens': 102, 'total_tokens': 736, 'input_token_detail